In [1]:
#Spark connection with S3 options
import os
import socket
from pyspark.sql import SparkSession

# Замените следующие значения на свои credentials
aws_access_key = "j612yzOjcIwYyPhVD14P"
aws_secret_key = "bVTEwJ7sJ0TboubM5wJcn2ieYrV91uXk3N9dJ457"
s3_bucket = "startde-datasets"
s3_endpoint_url = "https://s3.lab.karpov.courses"
 
APACHE_MASTER_IP = socket.gethostbyname("apache-spark-master-0.apache-spark-headless.apache-spark.svc.cluster.local")
APACHE_MASTER_URL = f"spark://{APACHE_MASTER_IP}:7077"
POD_IP = os.environ["MY_POD_IP"]
SPARK_APP_NAME = f"spark-{os.environ['HOSTNAME']}"
JARS = """/nfs/env/lib/python3.8/site-packages/pyspark/jars/clickhouse-native-jdbc-shaded-2.6.5.jar, 
/nfs/env/lib/python3.8/site-packages/pyspark/jars/hadoop-aws-3.3.4.jar,
/nfs/env/lib/python3.8/site-packages/pyspark/jars/aws-java-sdk-bundle-1.12.433.jar,
/nfs/env/lib/python3.8/site-packages/pyspark/jars/postgresql-42.7.4.jar
"""

MEM = "512m"
CORES = 1
 
spark = SparkSession.\
        builder.\
        appName(SPARK_APP_NAME).\
        master(APACHE_MASTER_URL).\
        config("spark.executor.memory", MEM).\
        config("spark.jars", JARS).\
        config("spark.executor.cores", CORES).\
        config("spark.driver.host", POD_IP).\
        config("spark.hadoop.fs.s3a.access.key", aws_access_key). \
        config("spark.hadoop.fs.s3a.secret.key", aws_secret_key). \
        config("fs.s3a.endpoint", s3_endpoint_url).  \
        config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem"). \
        config("spark.hadoop.fs.s3a.committer.name", "directory"). \
        config("spark.hadoop.fs.s3a.path.style.access", True). \
        config("spark.hadoop.fs.s3a.aws.credentials.provider", "org.apache.hadoop.fs.s3a.SimpleAWSCredentialsProvider"). \
        config("spark.hadoop.fs.s3a.connection.ssl.enabled", "false"). \
        getOrCreate()

26/02/08 11:00:29 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


In [2]:
from pyspark.sql.functions import *

# Read Source tables
products_table = spark.read.parquet("s3a://startde-datasets/shared/products.parquet")
sales_table = spark.read.parquet("s3a://startde-datasets/shared/sales.parquet")
sellers_table = spark.read.parquet("s3a://startde-datasets/shared/sellers.parquet")

26/02/08 11:00:33 WARN MetricsConfig: Cannot locate configuration: tried hadoop-metrics2-s3a-file-system.properties,hadoop-metrics2.properties


In [10]:
products_table.show()
sales_table.show()
sellers_table.show()

+----------+------------+-----+
|product_id|product_name|price|
+----------+------------+-----+
|         0|   product_0|   22|
|         1|   product_1|   30|
|         2|   product_2|   91|
|         3|   product_3|   37|
|         4|   product_4|  145|
|         5|   product_5|  128|
|         6|   product_6|   66|
|         7|   product_7|  145|
|         8|   product_8|   51|
|         9|   product_9|   44|
|        10|  product_10|   53|
|        11|  product_11|   13|
|        12|  product_12|  104|
|        13|  product_13|  102|
|        14|  product_14|   24|
|        15|  product_15|   14|
|        16|  product_16|   38|
|        17|  product_17|   72|
|        18|  product_18|   16|
|        19|  product_19|   46|
+----------+------------+-----+
only showing top 20 rows



+--------+----------+---------+----------+---------------+--------------------+
|order_id|product_id|seller_id|      date|num_pieces_sold|       bill_raw_text|
+--------+----------+---------+----------+---------------+--------------------+
|       1|         0|        0|2023-07-02|              7|itgntlqljxnCgkeax...|
|       2|         0|        0|2023-07-05|              9|ymkndxcgbyyqugbgp...|
|       3|         0|        0|2023-07-05|              7|qwbfznfqwfgkuXrnn...|
|       4|         0|        0|2023-07-03|              5|tybsrjpfucpitwqxk...|
|       5|         0|        0|2023-07-04|              3|crrjscajxlmugzmyg...|
|       6|         0|        0|2023-07-03|              1|blxycrmmzmaiarooj...|
|       7|         0|        0|2023-07-09|              8|ymidneomjcfnircqz...|
|       8|         0|        0|2023-07-02|              5|euavsmtstttcjxipl...|
|       9|         0|        0|2023-07-01|              7|jpwsylnygmxdfnswc...|
|      10|         0|        0|2023-07-0

In [11]:
 # Join sales and products tables

sales_products = sales_table.join(products_table, on="product_id")
 # Calculate revenue for each product in each order

sales_products = sales_products.withColumn("revenue", col("price") * col("num_pieces_sold"))

sales_products.show()

+----------+--------+---------+----------+---------------+--------------------+------------+-----+-------+
|product_id|order_id|seller_id|      date|num_pieces_sold|       bill_raw_text|product_name|price|revenue|
+----------+--------+---------+----------+---------------+--------------------+------------+-----+-------+
|         0|       1|        0|2023-07-02|              7|itgntlqljxnCgkeax...|   product_0|   22|    154|
|         0|       2|        0|2023-07-05|              9|ymkndxcgbyyqugbgp...|   product_0|   22|    198|
|         0|       3|        0|2023-07-05|              7|qwbfznfqwfgkuXrnn...|   product_0|   22|    154|
|         0|       4|        0|2023-07-03|              5|tybsrjpfucpitwqxk...|   product_0|   22|    110|
|         0|       5|        0|2023-07-04|              3|crrjscajxlmugzmyg...|   product_0|   22|     66|
|         0|       6|        0|2023-07-03|              1|blxycrmmzmaiarooj...|   product_0|   22|     22|
|         0|       7|        0|2023-0

In [12]:
sales_products_sellers = sales_products.join(sellers_table, on="seller_id")
sales_products_sellers.show()

+---------+----------+--------+----------+---------------+--------------------+------------+-----+-------+-----------+------------+
|seller_id|product_id|order_id|      date|num_pieces_sold|       bill_raw_text|product_name|price|revenue|seller_name|daily_target|
+---------+----------+--------+----------+---------------+--------------------+------------+-----+-------+-----------+------------+
|        0|         0|       1|2023-07-02|              7|itgntlqljxnCgkeax...|   product_0|   22|    154|   seller_0|        2500|
|        0|         0|       2|2023-07-05|              9|ymkndxcgbyyqugbgp...|   product_0|   22|    198|   seller_0|        2500|
|        0|         0|       3|2023-07-05|              7|qwbfznfqwfgkuXrnn...|   product_0|   22|    154|   seller_0|        2500|
|        0|         0|       4|2023-07-03|              5|tybsrjpfucpitwqxk...|   product_0|   22|    110|   seller_0|        2500|
|        0|         0|       5|2023-07-04|              3|crrjscajxlmugzmyg.

In [13]:
sales_products_sellers = sales_products_sellers.groupBy('date','seller_id','seller_name'). \
    agg(sum('revenue').alias('sum_revenue'))
sales_products_sellers.show()

+----------+---------+-----------+-----------+
|      date|seller_id|seller_name|sum_revenue|
+----------+---------+-----------+-----------+
|2023-07-03|        6|   seller_6|     490072|
|2023-07-02|        3|   seller_3|     483399|
|2023-07-07|        4|   seller_4|     413932|
|2023-07-10|        5|   seller_5|     443163|
|2023-07-10|        2|   seller_2|     472890|
|2023-07-02|        2|   seller_2|     445567|
|2023-07-09|        9|   seller_9|     481116|
|2023-07-06|        6|   seller_6|     426884|
|2023-07-07|        7|   seller_7|     461311|
|2023-07-04|        3|   seller_3|     462539|
|2023-07-09|        7|   seller_7|     453038|
|2023-07-08|        2|   seller_2|     452653|
|2023-07-08|        9|   seller_9|     455301|
|2023-07-08|        1|   seller_1|     449080|
|2023-07-04|        2|   seller_2|     469444|
|2023-07-01|        1|   seller_1|     479935|
|2023-07-08|        6|   seller_6|     469470|
|2023-07-01|        5|   seller_5|     447321|
|2023-07-03| 

In [14]:
sales_products_sellers = sales_products_sellers.groupBy('seller_id', 'seller_name'). \
    agg(avg('sum_revenue').alias('avg_daily_revenue')). \
    orderBy(desc('avg_daily_revenue')). \
    limit(3)
sales_products_sellers.show()

+---------+-----------+-----------------+
|seller_id|seller_name|avg_daily_revenue|
+---------+-----------+-----------------+
|        0|   seller_0|     2.29960082E7|
|        8|   seller_8|         468926.8|
|        3|   seller_3|         466907.3|
+---------+-----------+-----------------+



In [16]:
result = sales_products_sellers
pandas_df = result.toPandas()
pandas_df.to_parquet('result.parquet')

In [17]:
spark.stop()